In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

In [2]:
# Dummy Data Tabular [jam_tidur, aktivitas_fisik, jam_kerja]
X_tabular = np.array([
    [6.0, 1.0, 8.0],
    [3.0, 0.0, 12.0],
    [8.0, 2.0, 5.0]
])

In [3]:
# Dummy data teks catatan harian
X_text = np.array([
    "Hari ini capek banget, tapi semua kerjaan beres sih",
    "Mati rasa, tugas banyak yang belum selesai, dosen galak, pengen nyerah deh",
    "Santai banget hari ini, asik juga bisa mabar dan nongki"
], dtype=object)

In [4]:
# Label Target 0 (stress), 1 (normal), 2 (happy)
y = np.array([1, 0, 2])

# Lakukan one-hot encoding
y_one_hot = tf.keras.utils.to_categorical(y, num_classes=3)

In [5]:
# Layer untuk proses ekstraksi fitur secara otomatis
vectorize_layer = layers.TextVectorization(
    max_tokens=1000,
    output_mode='int',
    output_sequence_length=20
)

# Adaptasi X_test dengan vocab
vectorize_layer.adapt(X_text)

In [6]:
# Model khusus numerik (data tabular)
input_tab = layers.Input(shape=(3,), name="input_tab")
x1 = layers.Dense(16, activation='relu')(input_tab)
out_tab = layers.Dense(3, activation='softmax', name="out_tab")(x1)
model_tab = models.Model(input_tab, out_tab)

In [7]:
# Model khusus teks
input_txt = layers.Input(shape=(1,), dtype=tf.string, name="input_txt")
x2 = vectorize_layer(input_txt)
x2 = layers.Embedding(1000, 16)(x2)
x2 = layers.LSTM(16)(x2)
out_txt = layers.Dense(3, activation='softmax', name="out_txt")(x2)
model_txt = models.Model(input_txt, out_txt)

In [8]:
# Compile model secara terpisah

# Compile model tabular
model_tab.compile(optimizer='adam', 
                  loss='categorical_crossentropy', 
                  metrics=['accuracy'])

# Compile model teks
model_txt.compile(optimizer='adam', 
                  loss='categorical_crossentropy', 
                  metrics=['accuracy'])

In [9]:
# Train model secara terpisah

# Train model tabular
model_tab.fit(X_tabular, y_one_hot, epochs=10, verbose=1)

# Train model teks
model_txt.fit(X_text, y_one_hot, epochs=10, verbose=1)

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 531ms/step - accuracy: 0.3333 - loss: 1.0300
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.6667 - loss: 1.0079
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.6667 - loss: 0.9864
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.6667 - loss: 0.9656
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.6667 - loss: 0.9454
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.6667 - loss: 0.9259
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.6667 - loss: 0.9070
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.6667 - loss: 0.8888
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.6667 - loss: 0.8721
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.6667 - loss: 0.8562
Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.6667 - loss: 1.0983
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.6667 - loss: 1.0979
Epoc

In [10]:
# Ambil probabilitas kedua model
prob_tab = model_tab.predict(X_tabular)
prob_txt = model_txt.predict(X_text)

print("Probabilitas Tabular:\n", prob_tab)
print("Probabilitas Teks:\n", prob_txt)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step
Probabilitas Tabular:
 [[0.2666604  0.64770967 0.08562994]
 [0.24085204 0.74376297 0.015385  ]
 [0.19879052 0.2868947  0.5143147 ]]
Probabilitas Teks:
 [[0.3305039  0.33664945 0.3328466 ]
 [0.3346038  0.33080864 0.33458748]
 [0.3323527  0.3329673  0.33468002]]


## Probabilitas Prediksi Model Tabular

| Data ke- | Stress | Normal | Happy | Prediksi |
|---|---|---|---|---|
| 1 | 0.2667 | 0.6477 | 0.0856 | Normal |
| 2 | 0.2409 | 0.7438 | 0.0154 | Normal |
| 3 | 0.1988 | 0.2869 | 0.5143 | Happy |


## Probabilitas Prediksi Model Teks

| Data ke- | Stress | Normal | Happy | Prediksi |
|---|---|---|---|---|
| 1 | 0.3305 | 0.3366 | 0.3328 | Normal |
| 2 | 0.3346 | 0.3308 | 0.3346 | Stress |
| 3 | 0.3324 | 0.3330 | 0.3347 | Happy |


## Penjelasan Interpretasi

- Nilai probabilitas menunjukkan tingkat keyakinan model terhadap masing-masing kelas.
- Total probabilitas pada setiap baris ≈ 1.
- Prediksi akhir diambil dari probabilitas terbesar pada setiap data.
- Model tabular terlihat lebih yakin karena terdapat probabilitas dominan pada satu kelas.
- Model teks masih cukup bingung karena probabilitas antar kelas hampir sama (sekitar 0.33)

In [11]:
# Tentukan bobot (weighting) untuk lebih percaya model yang mana
w_tab = 0.4
w_txt = 0.6

In [12]:
# Gabungkan hasil (weight sum) & lakukan decision making
final_prob = (w_tab * prob_tab) + (w_txt * prob_txt)
final_class = np.argmax(final_prob, axis=1)

print("\nHasil Late Fusion:")
print(f"Prediksi Kelas Akhir: {final_class}")
print(f"Label Asli: {y}")


Hasil Late Fusion:
Prediksi Kelas Akhir: [1 1 2]
Label Asli: [1 0 2]
